# Elo strength model (Model 2 of 2)

A single strength rating per team, updated match-by-match — the parallel to the Dixon-Coles goal model. Same target, different mechanism, evaluated on the **same eval set** to go head-to-head with the DC benchmark **RPS 0.1646**. Elo is also one of the must-beat baselines.

**Build order:** split (identical to Model 1) → chronological rating engine → make it goal-capable (rating gap → λ's) → walk-forward eval on the eval set → compare to 0.1646.

In [12]:
import pandas as pd 
import numpy as np
from scipy.stats import poisson

In [13]:
results = pd.read_parquet('../data/interim/results_clean.parquet')
print(results.head())

        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1 1873-03-08   England  Scotland           4           2   Friendly   London   
2 1874-03-07  Scotland   England           2           1   Friendly  Glasgow   
3 1875-03-06   England  Scotland           2           2   Friendly   London   
4 1876-03-04  Scotland   England           3           0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  


In [14]:
# Same data scope + split as Model 1, so both models are evaluated on the identical eval set
model_df = results[results['date'].dt.year >= 1970].reset_index(drop=True)

# Leakage-safe split — boundaries mutually exclusive so no match is in two slices
wc2026_mask = (model_df['tournament'] == 'FIFA World Cup') & (model_df['date'].dt.year == 2026)

train_df = model_df[model_df['date'] < '2024-01-01']                   # Elo burns in on this
eval_df = model_df[(model_df['date'] >= '2024-01-01') & ~wc2026_mask]  # held-out test set (WC excluded)
wc2026_df = model_df[wc2026_mask]                                      # forecast target, held aside

print(f"train {len(train_df)} | eval {len(eval_df)} | wc2026 {len(wc2026_df)}")

train 38900 | eval 2543 | wc2026 79


## Elo rating engine

Every team starts at 1500. Processing matches **in date order**, before each game we compute the home team's expected score from the rating gap (an S-curve):

`E_home = 1 / (1 + 10^((R_away − R_home) / 400))`

Then compare to the actual result (`S` = 1 win / 0.5 draw / 0 loss) and nudge both ratings by `K·(S − E_home)`, zero-sum between the two teams. Beating a team you were *expected* to beat barely moves the rating; an upset moves it a lot. **K** (step size) is the tunable knob; **400** is the standard Elo scale (a 400-point lead ≈ 91% expected score).

In [15]:
def compute_elo_ratings(matches, k=20, base_rating=1500):
    ratings = {}   # team -> current rating (new teams start at base_rating)

    # Process matches in chronological order, nudging ratings after each game
    for _, row in matches.iterrows():
        home, away = row['home_team'], row['away_team']
        rating_home = ratings.get(home, base_rating)
        rating_away = ratings.get(away, base_rating)

        # Expected home score from the rating gap (S-curve, 0..1)
        expected_home = 1 / (1 + 10 ** ((rating_away - rating_home) / 400))

        # Actual home result: 1 win, 0.5 draw, 0 loss
        if row['home_score'] > row['away_score']:
            actual_home = 1.0
        elif row['home_score'] == row['away_score']:
            actual_home = 0.5
        else:
            actual_home = 0.0

        # Nudge both ratings by K x (surprise); zero-sum between the two teams
        change = k * (actual_home - expected_home)
        ratings[home] = rating_home + change
        ratings[away] = rating_away - change

    return ratings


# Sanity check: build ratings on train, inspect the top
elo_ratings = compute_elo_ratings(train_df)
for team, rating in sorted(elo_ratings.items(), key=lambda item: item[1], reverse=True)[:10]:
    print(f"{team:20s} {rating:.0f}")

Argentina            1973
France               1937
Brazil               1930
Spain                1916
England              1909
Belgium              1896
Portugal             1889
Netherlands          1877
Colombia             1877
Uruguay              1855


## Making Elo goal-capable

Elo natively gives one number (expected score), but RPS needs **W/D/L** and the simulator needs **goals**. So convert the rating gap into two expected-goal rates, then reuse Model 1's scoreline → W/D/L machinery:

- **supremacy** = (R_home − R_away + home_bonus) / D — the gap expressed as a goal difference
- **λ_home** = (μ_total + supremacy) / 2,  **λ_away** = (μ_total − supremacy) / 2

`D` (Elo points per goal), `home_bonus` (home advantage, off at neutral — same principle as Model 1), and `μ_total` (avg total goals) are the knobs, tuned on eval. A big favorite → large supremacy → λ_home ≫ λ_away; even teams → supremacy ≈ 0 → more draws.

In [16]:
def elo_lambdas(rating_home, rating_away, neutral, points_per_goal=150, home_bonus=65, mu_total=2.6):
    # Rating gap (+ home advantage unless neutral) -> expected goal supremacy
    home_advantage = 0 if neutral else home_bonus
    supremacy = (rating_home - rating_away + home_advantage) / points_per_goal

    # Split a typical total into the two sides according to the supremacy
    lambda_home = (mu_total + supremacy) / 2
    lambda_away = (mu_total - supremacy) / 2

    # Clip to stay positive (very lopsided games can push the underdog below 0)
    lambda_home = max(lambda_home, 0.05)
    lambda_away = max(lambda_away, 0.05)
    return lambda_home, lambda_away


# Sanity check on a neutral Brazil vs Argentina
lam_home, lam_away = elo_lambdas(elo_ratings['Brazil'], elo_ratings['Argentina'], neutral=True)
print(f"λ  Brazil {lam_home:.2f}  |  Argentina {lam_away:.2f}")

λ  Brazil 1.16  |  Argentina 1.44


In [17]:
# Same ordinal metric as Model 1 (reused so both models are scored identically)
def ranked_probability_score(predicted_probs, actual_outcome):
    cum_pred = np.cumsum(predicted_probs)
    cum_actual = np.cumsum(actual_outcome)
    n = len(predicted_probs)
    return np.sum((cum_pred - cum_actual) ** 2) / (n - 1)


def evaluate_elo(train_df, eval_df, k=20, points_per_goal=150, home_bonus=65, mu_total=2.6, max_goals=10):
    # Burn in ratings on train, then walk forward through eval
    ratings = compute_elo_ratings(train_df, k=k)

    rps_values = []
    for _, row in eval_df.iterrows():
        home, away = row['home_team'], row['away_team']
        rating_home = ratings.get(home, 1500)
        rating_away = ratings.get(away, 1500)

        # 1) PREDICT this match from current ratings (before seeing its result)
        lam_home, lam_away = elo_lambdas(rating_home, rating_away, row['neutral'],
                                         points_per_goal, home_bonus, mu_total)
        home_probs = poisson.pmf(np.arange(max_goals + 1), lam_home)
        away_probs = poisson.pmf(np.arange(max_goals + 1), lam_away)
        score_matrix = np.outer(home_probs, away_probs)
        probs = [np.tril(score_matrix, -1).sum(), np.trace(score_matrix), np.triu(score_matrix, 1).sum()]

        # 2) SCORE against the actual result
        if row['home_score'] > row['away_score']:
            actual = [1, 0, 0]
        elif row['home_score'] == row['away_score']:
            actual = [0, 1, 0]
        else:
            actual = [0, 0, 1]
        rps_values.append(ranked_probability_score(probs, actual))

        # 3) UPDATE ratings with the result, so the next match uses fresher ratings
        expected_home = 1 / (1 + 10 ** ((rating_away - rating_home) / 400))
        actual_home = actual[0] + 0.5 * actual[1]   # 1 win / 0.5 draw / 0 loss
        change = k * (actual_home - expected_home)
        ratings[home] = rating_home + change
        ratings[away] = rating_away - change

    return np.mean(rps_values)


print(evaluate_elo(train_df, eval_df))

0.16964440805015227
